In [4]:
import re
import urllib.parse
from collections import defaultdict
import pandas as pd
from rdflib import Graph, RDFS, URIRef

RELATIONS = {
    URIRef("http://cso.kmi.open.ac.uk/schema/cso#superTopicOf"): "super_topic_of", 
    URIRef("http://cso.kmi.open.ac.uk/schema/cso#contributesTo"): "contributes_to", 
    URIRef("http://cso.kmi.open.ac.uk/schema/cso#partOf"): "part_of", 
    URIRef("http://cso.kmi.open.ac.uk/schema/cso#relatedEquivalent"): "equivalent" 
}

DOMAIN_SEEDS = {
    "architecture_patterns": [
        "software architecture", "microservice", "monolith", "service mesh", 
        "event driven", "domain driven design", "hexagonal architecture", "clean architecture",
        "onion architecture", "layered architecture", "serverless", "design pattern", 
        "solid principles", "mvc", "mvvm", "cqrs", "event sourcing", "saga pattern", 
        "circuit breaker", "sidecar pattern", "bff", "coupling", "cohesion"
    ],
    "system_design_tradeoffs": [
        "system design", "scalability", "availability", "reliability", 
        "fault tolerance", "load balancing", "caching", "sharding", 
        "replication", "consistency", "cap theorem", "distributed system", 
        "concurrency", "throughput", "latency", "api gateway", "rate limiting",
        "message queue", "pub sub", "service discovery", "consensus algorithm",
        "partitioning", "idempotency", "backpressure"
    ],
    "development_lifecycle": [
        "agile", "scrum", "kanban", "test driven development", "tdd",
        "ci/cd", "continuous integration", "continuous deployment", 
        "code review", "refactoring", "technical debt", "git", 
        "software quality", "devops", "site reliability engineering", "sre"
    ],
    "data_storage_strategy": [
        "relational database", "nosql", "sql", "acid", "eventual consistency",
        "transaction", "indexing", "orm", "data warehouse", "data lake",
        "etl", "stream processing", "kafka", "redis", "elasticsearch",
        "graph database", "document store", "key value store"
    ],
    "cloud_infrastructure": [
        "cloud computing", "docker", "kubernetes", "containerization", 
        "infrastructure as code", "terraform", "ansible", "serverless computing",
        "auto scaling", "multi tenancy", "virtualization", "hybrid cloud"
    ],
    "programming_paradigms": [
        "object oriented", "functional programming", "reactive programming",
        "asynchronous programming", "memory management", "garbage collection",
        "multithreading", "parallel computing", "type system", "metaprogramming"
    ],
    "security_identity": [
        "authentication", "authorization", "oauth", "jwt", "encryption",
        "vulnerability", "threat modeling", "zero trust", "identity management",
        "tls", "ssl", "cors", "cross site scripting"
    ]
}

ALL_SEEDS = [s for seeds in DOMAIN_SEEDS.values() for s in seeds]
SEED_TO_DOMAIN = {s: d for d, seeds in DOMAIN_SEEDS.items() for s in seeds}

def normalize_id(label: str) -> str:
    s = label.lower().strip()
    s = re.sub(r"\([^)]*\)", "", s) 
    s = re.sub(r"[^a-z0-9\s\-]", "", s) 
    s = re.sub(r"[\s\-]+", "_", s) 
    s = re.sub(r"_+", "_", s) 
    return s.strip("_")

def get_domain(label: str) -> str:
    low = label.lower()
    for seed in ALL_SEEDS:
        if len(seed) <= 4:
            if re.search(rf"\b{re.escape(seed)}\b", low): return SEED_TO_DOMAIN[seed]
        else:
            if seed in low: return SEED_TO_DOMAIN[seed]
    return None

def get_label(g, uri: URIRef) -> str:
    label = g.value(uri, RDFS.label)
    if label: return str(label)
    return urllib.parse.unquote(str(uri).split("/")[-1]).replace("_", " ")

def is_academic_noise(label: str) -> bool:
    noise_words = [
        'analysis', 'evaluation', 'study', 'approach', 'framework', 
        'algorithm for', 'based', 'investigation', 'empirical'
    ]
    lbl_lower = label.lower()
    return any(word in lbl_lower for word in noise_words)


def run_pipeline(input_path: str):
    g = Graph()
    print("Loading CSO Ontology (this may take a few minutes)...")
    g.parse(input_path, format="nt")
    print(f"Graph loaded with {len(g)} triples.")
    
    topic_to_domain = {}
    uri_to_label = {}
    
    print("\nPass 1: Finding anchor topics from seeds...")
    all_uris = set(g.subjects()).union({o for o in g.objects() if isinstance(o, URIRef)})
    
    for uri in all_uris:
        lbl = get_label(g, uri)
        if not is_academic_noise(lbl):
            dom = get_domain(lbl)
            if dom:
                topic_to_domain[str(uri)] = dom
                uri_to_label[str(uri)] = lbl

    print("\nPass 2: Expanding graph via superTopicOf relations...")
    super_topic_uri = URIRef("http://cso.kmi.open.ac.uk/schema/cso#superTopicOf")
    
    anchors = list(topic_to_domain.keys())
    
    for parent_uri_str in anchors:
        parent_domain = topic_to_domain[parent_uri_str]
        parent_uri = URIRef(parent_uri_str)
        
        for _, _, child_uri in g.triples((parent_uri, super_topic_uri, None)):
            if isinstance(child_uri, URIRef):
                child_str = str(child_uri)
                if child_str not in topic_to_domain:
                    child_lbl = get_label(g, child_uri)
                    if not is_academic_noise(child_lbl):
                        topic_to_domain[child_str] = parent_domain
                        uri_to_label[child_str] = child_lbl

    print("\nPass 3: Extracting edges between identified topics...")
    edge_list = []
    active_uris = set()

    for s, p, o in g:
        if p in RELATIONS and str(s) in topic_to_domain and str(o) in topic_to_domain:
            s_uri, o_uri = str(s), str(o)
            s_id = normalize_id(uri_to_label[s_uri])
            o_id = normalize_id(uri_to_label[o_uri])
            
            if s_id != o_id:
                edge_list.append({
                    "source": s_id,
                    "target": o_id,
                    "rel_type": RELATIONS[p]
                })
                active_uris.update([s_uri, o_uri])

    print("\nPass 4: Formatting and saving to CSV...")
    nodes_data = [
        {
            "topic_id": normalize_id(uri_to_label[uri]),
            "name": uri_to_label[uri].title(), 
            "domain": topic_to_domain[uri]
        }
        for uri in active_uris
    ]

    nodes_df = pd.DataFrame(nodes_data).drop_duplicates(subset=['topic_id'])
    edges_df = pd.DataFrame(edge_list).drop_duplicates()

    nodes_df.to_csv("neo4j_nodes.csv", index=False)
    edges_df.to_csv("neo4j_edges.csv", index=False)
    
    print(f"\nPipeline Complete!")
    print(f"Final Graph Size: {len(nodes_df)} Nodes, {len(edges_df)} Edges.")    

ModuleNotFoundError: No module named 'rdflib'

In [ ]:
run_pipeline(r"data\CSO.3.5.nt")

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
import uuid

In [ ]:
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [5]:
def seed_neo4j(nodes_csv: str, edges_csv: str):
    nodes_df = pd.read_csv(nodes_csv)
    edges_df = pd.read_csv(edges_csv)

    def _merge_nodes(tx, df):
        query = """
        UNWIND $rows AS row
        MERGE (t:Topic {topic_id: row.topic_id})
        SET t.name = row.name, t.domain = row.domain
        """
        tx.run(query, rows=df.to_dict('records'))

    def _merge_edges(tx, df):
        for _, row in df.iterrows():
            query = f"""
            MATCH (source:Topic {{topic_id: $source_id}})
            MATCH (target:Topic {{topic_id: $target_id}})
            MERGE (source)-[r:{row['rel_type'].upper()}]->(target)
            """
            tx.run(query, source_id=row['source'], target_id=row['target'])

    with driver.session() as session:
        print("Seeding Neo4j Nodes...")
        session.execute_write(_merge_nodes, nodes_df)
        print("Seeding Neo4j Edges...")
        session.execute_write(_merge_edges, edges_df)
        print("Neo4j Seeding Complete!")

In [6]:
seed_neo4j(nodes_csv="neo4j_nodes.csv" , edges_csv="neo4j_edges.csv")

Seeding Neo4j Nodes...
Seeding Neo4j Edges...
Neo4j Seeding Complete!
